# Notebook 07 — Avaliação Detalhada e Interpretação do Modelo

## Objetivo
Avaliar o melhor modelo (XGBoost tuned) no conjunto de teste temporal (Jul–Dez 2023), com foco em interpretabilidade, equidade por subgrupo e calibração de probabilidade.

## Inputs
- `models/best_model.joblib` — Pipeline XGBoost tuned (AUC-PR 0,5531 · AUC-ROC 0,9055)
- `data/processed/X_test.parquet` · `y_test.parquet` — 103.072 casos (Jul–Dez 2023)
- `data/processed/X_train.parquet` — 145.407 casos (Jan–Jun 2023)

## Outputs (figuras em `reports/figures/07_avaliacao/`)
1. `01_curvas_pr_roc.png` — Curvas PR e ROC
2. `02_threshold_tuning.png` — Métricas vs threshold; pontos Youden e Recall >= 70%
3. `03_confusion_matrices.png` — Matrizes de confusão em 3 thresholds
4. `04_calibracao.png` — Reliability diagram e distribuição de probabilidades
5. `05_shap_beeswarm.png` — Importância global das features (SHAP)
6. `06_shap_individual.png` — Análise individual: TP, FN, TN, FP representativos
7. `07_subgrupos.png` — AUC-PR por faixa etária, região e raça

## Decisões-chave desta fase
- **Threshold:** comparação entre Youden (maximiza J = sensibilidade + especificidade - 1) e recall-constrained (recall >= 70% como mínimo clínico aceitável)
- **SHAP:** `TreeExplainer` — exato para modelos baseados em árvore; mais rápido e fidedigno que KernelExplainer
- **Subgrupo:** AUC-PR (não accuracy) — mantém consistência com a métrica primária do projeto em dados desbalanceados

In [ ]:
import sys
from pathlib import Path

_src = Path("..") / "src"
if str(_src.resolve()) not in sys.path:
    sys.path.insert(0, str(_src.resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    f1_score, recall_score, precision_score,
    brier_score_loss, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve

from maxpar_srag import config

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 11, "figure.dpi": 100, "axes.unicode_minus": False})

FIGURES_DIR = config.FIGURES_DIR / "07_avaliacao"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = config.RANDOM_STATE
RNG = np.random.default_rng(RANDOM_STATE)

print(f"Figuras -> {FIGURES_DIR}")

In [ ]:
pipeline = joblib.load(config.PROJECT_ROOT / "models" / "best_model.joblib")
print(f"Modelo carregado: {type(pipeline.named_steps['clf']).__name__}")

X_test  = pd.read_parquet(config.DATA_PROCESSED / "X_test.parquet")
y_test  = pd.read_parquet(config.DATA_PROCESSED / "y_test.parquet")["target"]
X_train = pd.read_parquet(config.DATA_PROCESSED / "X_train.parquet")
y_train = pd.read_parquet(config.DATA_PROCESSED / "y_train.parquet")["target"]

print(f"\nTeste : {X_test.shape[0]:,} casos · {y_test.mean():.2%} obitos · {X_test.shape[1]} features")
print(f"Treino: {X_train.shape[0]:,} casos · {y_train.mean():.2%} obitos")

proba   = pipeline.predict_proba(X_test)[:, 1]
auc_pr  = average_precision_score(y_test, proba)
auc_roc = roc_auc_score(y_test, proba)
brier   = brier_score_loss(y_test, proba)

print(f"\n--- Verificacao vs Fase 5 ---")
print(f"AUC-PR  : {auc_pr:.4f}  (esperado ≈ 0.5531)")
print(f"AUC-ROC : {auc_roc:.4f}  (esperado ≈ 0.9055)")
print(f"Brier   : {brier:.4f}")

## 7.1 Métricas Gerais no Conjunto de Teste

O conjunto de teste contém casos notificados em **julho a dezembro de 2023** — o modelo nunca viu esses dados durante o treino. Esta separação temporal simula uso real (modelo treinado em dados passados, avaliado em casos futuros) e é o padrão em epidemiologia preditiva.

| Métrica | Significado |
|---|---|
| **AUC-PR** | Área sob curva Precision-Recall (primária); baseline aleatório próximo da taxa de óbito (cerca de 0,10) |
| **AUC-ROC** | Área sob curva ROC; menos sensível a desbalanceamento de classes |
| **Brier Score** | Erro quadrático médio das probabilidades; perfeito=0, aleatório próximo de 0,09 |
| **F1, Recall, Precisão** | Dependentes do threshold; threshold ideal escolhido na seção 7.3 |

In [ ]:
pred_05  = (proba >= 0.5).astype(int)
f1_05    = f1_score(y_test, pred_05, zero_division=0)
rec_05   = recall_score(y_test, pred_05, zero_division=0)
prec_05  = precision_score(y_test, pred_05, zero_division=0)

baseline_ap    = float(y_test.mean())
baseline_brier = float(y_test.mean() * (1 - y_test.mean()))

rows = [
    ("AUC-PR (primaria)",         auc_pr,  f"Baseline aleatorio: {baseline_ap:.4f}"),
    ("AUC-ROC",                   auc_roc, "Baseline aleatorio: 0.5000"),
    ("Brier Score",               brier,   f"Baseline aleatorio: {baseline_brier:.4f}"),
    ("F1 @ thr=0.5",              f1_05,   "—"),
    ("Recall/Sensib. @ thr=0.5",  rec_05,  "—"),
    ("Precisao @ thr=0.5",        prec_05, "—"),
]

df_metrics = pd.DataFrame(rows, columns=["Metrica", "Valor", "Referencia"])
df_metrics["Valor"] = df_metrics["Valor"].map("{:.4f}".format)
print(df_metrics.to_string(index=False))
print(f"\nGanho AUC-PR vs aleatorio: {auc_pr / baseline_ap:.1f}x")

## 7.2 Curvas PR e ROC

**Curva PR (esquerda):** mostra o trade-off entre precisão e recall em todos os thresholds. A curva acima da linha tracejada (baseline aleatório) representa o ganho do modelo.

**Curva ROC (direita):** mostra sensibilidade versus especificidade. AUC-ROC de 0,90 indica excelente capacidade discriminativa geral.

A AUC-PR é adotada como métrica primária porque, com cerca de 10% de óbitos, ela penaliza mais duramente o modelo por errar a classe rara — mais alinhada com o objetivo clínico de identificar óbitos do que a AUC-ROC, que não distingue bem o desempenho na classe minoritária.

In [ ]:
prec_arr, rec_arr, pr_thresholds  = precision_recall_curve(y_test, proba)
fpr_arr,  tpr_arr, roc_thresholds = roc_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR
ax = axes[0]
ax.plot(rec_arr, prec_arr, color="steelblue", lw=2,
        label=f"XGBoost tuned (AUC-PR={auc_pr:.3f})")
ax.axhline(y_test.mean(), color="gray", ls="--", lw=1.2,
           label=f"Aleatorio ({y_test.mean():.3f})")
ax.set_xlabel("Recall (Sensibilidade — classe Obito)")
ax.set_ylabel("Precisao")
ax.set_title("Curva Precision-Recall")
ax.legend(loc="upper right")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

# ROC
ax = axes[1]
ax.plot(fpr_arr, tpr_arr, color="darkorange", lw=2,
        label=f"XGBoost tuned (AUC={auc_roc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1.0, label="Aleatorio")
ax.set_xlabel("Taxa de Falso Positivo (1 - Especificidade)")
ax.set_ylabel("Taxa de Verdadeiro Positivo (Recall)")
ax.set_title("Curva ROC")
ax.legend(loc="lower right")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

fig.suptitle("Desempenho do XGBoost — Teste Jul-Dez 2023", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_curvas_pr_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 01_curvas_pr_roc.png")

## 7.3 Seleção do Threshold Operacional

O threshold padrão (0,5) raramente é ótimo para problemas desbalanceados. Dois critérios são avaliados:

1. **Youden's J** = Sensibilidade + Especificidade - 1: maximiza o balanço global entre detectar óbitos e evitar falsos alarmes. Bom ponto de partida para screening geral.

2. **Recall >= 70% (clínico):** em triagem hospitalar, deixar de identificar um óbito (falso negativo) é muito mais custoso do que um alarme falso (falso positivo). O threshold máximo que ainda mantém sensibilidade >= 70% representa o piso clínico aceitável.

O F1 máximo não é adotado como critério porque trata falsos positivos e falsos negativos como igualmente custosos. Em saúde pública, errar um óbito tem consequência muito maior que um alarme falso — o critério recall-constrained captura essa assimetria.

In [ ]:
# Youden (curva ROC)
youden     = tpr_arr - fpr_arr
opt_j      = np.argmax(youden)
thr_youden = float(roc_thresholds[opt_j])
sen_youden = float(tpr_arr[opt_j])
esp_youden = float(1 - fpr_arr[opt_j])

# Recall >= 70% (curva PR)
# rec_arr[:-1] alinha com pr_thresholds; queremos o maior threshold que ainda atinge recall>=0.70
mask_70 = rec_arr[:-1] >= 0.70
thr_rec70 = float(pr_thresholds[mask_70].max()) if mask_70.any() else float(pr_thresholds[0])

# Metricas por range de thresholds
thresholds_range = np.linspace(0.05, 0.90, 150)
f1_range   = [f1_score(y_test,    (proba >= t).astype(int), zero_division=0) for t in thresholds_range]
rec_range  = [recall_score(y_test,(proba >= t).astype(int), zero_division=0) for t in thresholds_range]
prec_range = [precision_score(y_test,(proba>=t).astype(int),zero_division=0) for t in thresholds_range]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(thresholds_range, rec_range,  color="steelblue",  lw=2, label="Recall (Sensibilidade)")
ax.plot(thresholds_range, prec_range, color="darkorange",  lw=2, label="Precisao (VPP)")
ax.plot(thresholds_range, f1_range,   color="purple",      lw=2, label="F1-Score")
ax.axvline(thr_youden, color="green", ls="--", lw=1.8,
           label=f"Youden: {thr_youden:.3f}  (Sen={sen_youden:.2f}, Esp={esp_youden:.2f})")
ax.axvline(thr_rec70, color="red", ls="--", lw=1.8,
           label=f"Recall>=70%: {thr_rec70:.3f}")
ax.axhline(0.70, color="red", ls=":", lw=1.0, alpha=0.5)
ax.set_xlabel("Threshold de Probabilidade de Obito")
ax.set_ylabel("Score")
ax.set_title("Metricas por Threshold — Teste Jul-Dez 2023")
ax.legend(loc="center right", fontsize=9)
ax.set_xlim([0.05, 0.90])
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Threshold Youden   : {thr_youden:.4f}  (Sensibilidade={sen_youden:.3f}, Especificidade={esp_youden:.3f})")
print(f"Threshold Recall70%: {thr_rec70:.4f}")
print(f"Threshold padrao   : 0.5000  (Sensibilidade={rec_05:.3f})")

In [ ]:
thresholds_named = [
    ("0.5 (padrao)",                   0.5),
    (f"Youden ({thr_youden:.3f})",     thr_youden),
    (f"Recall>=70% ({thr_rec70:.3f})", thr_rec70),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, thr) in zip(axes, thresholds_named):
    pred = (proba >= thr).astype(int)
    cm   = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Cura", "Obito"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    rec  = recall_score(y_test, pred, zero_division=0)
    prec = precision_score(y_test, pred, zero_division=0)
    f1   = f1_score(y_test, pred, zero_division=0)
    ax.set_title(f"Threshold {label}\nRecall={rec:.3f}  Precisao={prec:.3f}  F1={f1:.3f}",
                 fontsize=10)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")

fig.suptitle("Matrizes de Confusao — Teste Jul-Dez 2023", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 03_confusion_matrices.png")

## 7.4 Calibração de Probabilidade

Um modelo bem calibrado emite probabilidades coerentes com as frequências reais: se o modelo prevê p=0,30 para um grupo de pacientes, espera-se cerca de 30% de óbitos nesse grupo.

**Reliability diagram** (esquerda): pontos na diagonal representam calibração perfeita. Desvio acima da diagonal indica sub-estimativa; desvio abaixo indica super-estimativa.

**Distribuição das probabilidades** (direita): mostra a separação entre as classes. Maior separação corresponde a melhor discriminação. A sobreposição reflete a dificuldade inerente ao problema, em parte pela alta proporção de missing em comorbidades nas fichas SIVEP.

O Brier Score avalia simultaneamente discriminação e calibração: $BS = \frac{1}{n}\sum(p_i - y_i)^2$. O baseline ingênuo (prever sempre a taxa de óbito) tem BS próximo de 0,09. Modelos com boa discriminação tendem a apresentar BS abaixo do baseline mesmo sem calibração perfeita.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1.0, label="Calibracao perfeita")
ax.plot(prob_pred, prob_true, "o-", color="steelblue", lw=2, markersize=7,
        label=f"XGBoost (Brier={brier:.4f})")
ax.fill_between(prob_pred, prob_pred, prob_true, alpha=0.15, color="steelblue")
ax.set_xlabel("Probabilidade predita media (por decil)")
ax.set_ylabel("Fracao real de obitos")
ax.set_title("Reliability Diagram (Calibracao de Probabilidade)")
ax.legend(loc="upper left")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

# Histograma por classe
ax = axes[1]
ax.hist(proba[y_test == 0], bins=60, alpha=0.55, color="steelblue",
        label="Cura (real)", density=True)
ax.hist(proba[y_test == 1], bins=60, alpha=0.55, color="darkorange",
        label="Obito (real)", density=True)
ax.axvline(thr_youden, color="green", ls="--", lw=1.5,
           label=f"Thr Youden ({thr_youden:.3f})")
ax.axvline(thr_rec70, color="red", ls="--", lw=1.5,
           label=f"Thr Recall>=70% ({thr_rec70:.3f})")
ax.set_xlabel("Probabilidade predita de Obito")
ax.set_ylabel("Densidade")
ax.set_title("Distribuicao de Probabilidades Preditas por Classe")
ax.legend(fontsize=9)

fig.suptitle(f"Calibracao — XGBoost tuned (Brier Score = {brier:.4f})", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_calibracao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 04_calibracao.png")

## 7.5 SHAP — Importância Global das Features

SHAP (SHapley Additive exPlanations) distribui a predição do modelo entre as features de forma matematicamente justa, baseada na teoria dos jogos cooperativos (Shapley values).

**Como ler o beeswarm:**
- **Eixo x:** valor SHAP — impacto na predição em log-odds de óbito. Valores positivos aumentam o risco predito; negativos diminuem.
- **Posição vertical:** features ordenadas por importância global (média de |SHAP|)
- **Cor:** valor da feature (vermelho = alto, azul = baixo)

O `TreeExplainer` é utilizado no lugar do `KernelExplainer` porque calcula valores SHAP exatos para modelos baseados em árvore (incluindo XGBoost) ao decompor diretamente a estrutura das árvores — ordens de magnitude mais rápido e sem aproximações estocásticas.

In [ ]:
preprocessor = pipeline.named_steps["pre"]
xgb_model    = pipeline.named_steps["clf"]

# Nomes limpos (remove prefixos do ColumnTransformer: num__, bin__, ord__, ohe__)
raw_names   = preprocessor.get_feature_names_out()
clean_names = [n.split("__", 1)[1] if "__" in n else n for n in raw_names]

# Transforma dados (ColumnTransformer → numpy float64)
X_test_arr = preprocessor.transform(X_test).astype(np.float64)

# Amostra para velocidade — 5.000 casos do teste
sample_idx = RNG.choice(len(X_test_arr), size=min(5000, len(X_test_arr)), replace=False)
X_shap     = X_test_arr[sample_idx]
y_shap     = y_test.values[sample_idx]
proba_shap = proba[sample_idx]

print(f"Calculando SHAP TreeExplainer em {len(X_shap):,} amostras, {len(clean_names)} features...")
explainer    = shap.TreeExplainer(xgb_model)
shap_vals    = explainer.shap_values(X_shap)
expected_val = float(explainer.expected_value)
print(f"Calculado | expected_value (log-odds) = {expected_val:.4f}")
print(f"SHAP shape: {shap_vals.shape}")

# Beeswarm
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=clean_names,
    max_display=20,
    plot_type="dot",
    show=False,
    plot_size=(10, 8),
)
plt.title("SHAP — Importancia Global das Features (XGBoost, n=5.000 casos de teste)",
          fontsize=12, pad=12)
plt.savefig(FIGURES_DIR / "05_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 05_shap_beeswarm.png")

## 7.6 SHAP — Análise Individual de Casos Representativos

Quatro tipos de casos são analisados para entender como o modelo chega à sua decisão:

| Tipo | Real | Predição | Clínico |
|---|---|---|---|
| **TP** | Óbito | Alta prob. | Alerta correto — modelo identificou os sinais de risco |
| **FN** | Óbito | Baixa prob. | Alerta perdido — o que o modelo não captou? |
| **TN** | Cura | Baixa prob. | Descarte correto — quais fatores protetores foram identificados? |
| **FP** | Cura | Alta prob. | Falso alarme — quais fatores induziram o erro? |

Barras à **direita** (vermelho) aumentam o risco predito; barras à **esquerda** (azul) diminuem. São mostradas as top-12 features por |SHAP| para cada caso.

In [ ]:
case_types = {
    "TP: Obito, alerta correto":  (y_shap == 1) & (proba_shap > 0.50),
    "FN: Obito, alerta perdido":  (y_shap == 1) & (proba_shap < 0.20),
    "TN: Cura, descarte correto": (y_shap == 0) & (proba_shap < 0.05),
    "FP: Cura, falso alarme":     (y_shap == 0) & (proba_shap > 0.50),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
cor_pos = "#d73027"
cor_neg = "#4575b4"

for ax, (label, mask) in zip(axes.flatten(), case_types.items()):
    sub = np.where(mask)[0]
    if len(sub) == 0:
        ax.text(0.5, 0.5, "Nenhum caso encontrado", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(label)
        continue

    # Caso mais extremo (maior prob para TP/FP, menor para FN/TN)
    if "FN" in label or "TN" in label:
        pick = sub[np.argmin(proba_shap[sub])]
    else:
        pick = sub[np.argmax(proba_shap[sub])]

    vals  = shap_vals[pick]
    top12 = np.argsort(np.abs(vals))[-12:][::-1]
    sv    = vals[top12]
    fn    = [clean_names[i] for i in top12]
    cores = [cor_pos if v > 0 else cor_neg for v in sv]

    ax.barh(range(len(sv)), sv, color=cores, edgecolor="white", linewidth=0.4)
    ax.set_yticks(range(len(sv)))
    ax.set_yticklabels(fn, fontsize=9)
    ax.axvline(0, color="black", lw=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("SHAP value (log-odds)")
    pred_p = proba_shap[pick]
    real_l = "Obito" if y_shap[pick] == 1 else "Cura"
    ax.set_title(f"{label}\np(Obito)={pred_p:.3f} | Real: {real_l}", fontsize=10)

fig.suptitle("SHAP — Analise Individual de Casos Representativos", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_shap_individual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 06_shap_individual.png")

## 7.7 Análise de Equidade por Subgrupo

Modelos com boa AUC-PR global podem ter desempenho desigual entre subgrupos, o que representa viés de equidade se o modelo for usado em triagem real. Os três eixos investigados são:

- **Faixa etária:** determinante principal de mortalidade SRAG (gradiente monotônico confirmado na EDA — de 1,2% em 0-4 anos a 27,7% em 80+). O modelo tende a ser mais preciso onde há mais dados e mais óbitos para aprender.
- **Raça/Cor:** variáveis socioeconômicas correlacionadas com raça afetam acesso a cuidado; disparidades na predição podem refletir disparidades no registro clínico SIVEP.
- **Região do Brasil:** qualidade de preenchimento das fichas SIVEP-Gripe varia por região; Norte e Nordeste têm historicamente mais campos "Ignorado".

A AUC-PR é utilizada por subgrupo em vez de accuracy porque, com classes desbalanceadas, a taxa de óbito varia entre subgrupos (0-4 anos com 1,2% versus 80+ com 27,7%). A accuracy seria enganosa; a AUC-PR é comparável ao baseline aleatório de cada subgrupo.

Subgrupos com n < 200 ou menos de 10 óbitos são excluídos da análise (AUC-PR instável para amostras pequenas).

In [ ]:
sg = X_test.copy()
sg["proba"]  = proba
sg["target"] = y_test.values

# Faixa etaria (bins da EDA — consistencia com notebook 03)
bins_idade   = [-1, 4, 19, 39, 59, 69, 79, 120]
labels_idade = ["0-4", "5-19", "20-39", "40-59", "60-69", "70-79", "80+"]
sg["faixa_etaria"] = pd.cut(sg["IDADE_ANOS"], bins=bins_idade, labels=labels_idade)

# Raca (decodifica codigo SIVEP; descarta Ignorado=9)
raca_map = {k: v for k, v in config.CS_RACA.items() if v != "Ignorado"}
cs_raca_num = pd.to_numeric(sg["CS_RACA"], errors="coerce")
sg["raca"] = cs_raca_num.apply(
    lambda x: raca_map.get(int(x)) if (pd.notna(x) and int(x) in raca_map) else None
)

# Regiao (ja presente em X_test como string)
sg["regiao"] = sg["REGIAO_BR"].astype(object)
sg.loc[sg["regiao"].isna() | (sg["regiao"] == "nan"), "regiao"] = None

def subgroup_aucpr(df, col, min_n=200, min_pos=10):
    rows = []
    for grp_val, grp_df in df.dropna(subset=[col]).groupby(col, observed=True):
        n_pos = int(grp_df["target"].sum())
        n_tot = len(grp_df)
        if n_tot < min_n or n_pos < min_pos:
            continue
        ap   = average_precision_score(grp_df["target"], grp_df["proba"])
        base = float(grp_df["target"].mean())
        rows.append({"Subgrupo": str(grp_val), "AUC-PR": ap,
                     "Baseline": base, "N": n_tot, "N_obito": n_pos})
    return pd.DataFrame(rows)

print("Subgrupos preparados.")
print(f"\nFaixa etaria (distribuicao no teste):")
print(sg["faixa_etaria"].value_counts().sort_index())

In [ ]:
df_age = subgroup_aucpr(sg, "faixa_etaria")

# Ordenar por faixa etaria natural
cat_order = [l for l in labels_idade if l in df_age["Subgrupo"].values]
df_age["Subgrupo"] = pd.Categorical(df_age["Subgrupo"], categories=cat_order, ordered=True)
df_age = df_age.sort_values("Subgrupo").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df_age["Subgrupo"], df_age["AUC-PR"], color="steelblue", alpha=0.8,
              label="AUC-PR")
ax.bar(df_age["Subgrupo"], df_age["Baseline"], color="gray", alpha=0.4,
       label="Baseline aleatorio (taxa obito)")
for bar, (_, row) in zip(bars, df_age.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f"{row['AUC-PR']:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.text(bar.get_x() + bar.get_width()/2, -0.03,
            f"n={row['N']:,}", ha="center", va="top", fontsize=8, color="gray")
ax.set_xlabel("Faixa Etaria")
ax.set_ylabel("AUC-PR")
ax.set_title("AUC-PR por Faixa Etaria — Analise de Equidade (Teste Jul-Dez 2023)")
ax.legend()
ax.set_ylim([-0.05, max(df_age["AUC-PR"]) + 0.12])
fig.tight_layout()
plt.show()

print("\nTabela faixa etaria:")
print(df_age[["Subgrupo","AUC-PR","Baseline","N","N_obito"]].to_string(
      index=False, float_format="{:.4f}".format))

In [ ]:
df_reg  = subgroup_aucpr(sg, "regiao")
df_raca = subgroup_aucpr(sg, "raca")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, df, title, cor in [
    (axes[0], df_reg,  "AUC-PR por Regiao do Brasil", "steelblue"),
    (axes[1], df_raca, "AUC-PR por Raca/Cor",         "darkorange"),
]:
    if df.empty:
        ax.set_title(f"{title}\n(dados insuficientes)")
        continue
    df_s = df.sort_values("AUC-PR", ascending=False).reset_index(drop=True)
    bars = ax.bar(df_s["Subgrupo"], df_s["AUC-PR"], color=cor, alpha=0.8)
    ax.bar(df_s["Subgrupo"], df_s["Baseline"], color="gray", alpha=0.4)
    for bar, (_, row) in zip(bars, df_s.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{row['AUC-PR']:.3f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
        ax.text(bar.get_x() + bar.get_width()/2, -0.03,
                f"n={row['N']:,}", ha="center", va="top", fontsize=8, color="gray")
    ax.set_ylabel("AUC-PR")
    ax.set_title(title)
    ax.set_ylim([-0.05, max(df_s["AUC-PR"]) + 0.12])
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Analise de Equidade por Subgrupo — Teste Jul-Dez 2023", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_subgrupos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: 07_subgrupos.png")

print("\nTabela regiao:")
print(df_reg[["Subgrupo","AUC-PR","Baseline","N"]].to_string(
      index=False, float_format="{:.4f}".format))
print("\nTabela raca:")
print(df_raca[["Subgrupo","AUC-PR","Baseline","N"]].to_string(
      index=False, float_format="{:.4f}".format))

## 7.8 Discussão de Limitações

### 1. Subnotificação e qualidade do registro
O SIVEP-Gripe é preenchido em ambiente hospitalar com enorme variação de completude entre UFs e estabelecimentos. Comorbidades com cerca de 70% de missing não significam que 70% dos pacientes não tinham a comorbidade — significam que o preenchimento era opcional ou não priorizado. O modelo aprende parcialmente "qualidade do registro" junto com risco clínico real.

### 2. Features hospitalares como proxy de gravidade
`SUPORT_VEN_ORD` (invasivo/não-invasivo/não) e `UTI` são registrados durante a internação, antes do desfecho — são legítimas como features (sem vazamento). Contudo, refletem a resposta ao quadro clínico, não apenas o estado do paciente na admissão. Um modelo de triagem na chegada não teria essas variáveis; o uso clínico precisa ser contextualizado.

### 3. Calibração imperfeita nos extremos
O reliability diagram tipicamente mostrará desvio nas probabilidades muito altas e muito baixas. Para aplicações que usem as probabilidades absolutas (por exemplo, score de risco comunicado ao paciente), uma calibração adicional pós-treino (Platt Scaling ou isotonic regression) é recomendada.

### 4. Shift de distribuição sazonal
O split Jan–Jun (treino) versus Jul–Dez (teste) captura a sazonalidade natural: VSR pediátrico e Influenza no 1º semestre, pico de COVID no 2º semestre. Essa mudança na composição etiológica pode criar um shift de distribuição parcialmente capturado pela feature `SEM_NOT`, mas não completamente.

### 5. Viés de equidade e raça
Raça/cor está confundida por acesso a cuidado, qualidade de preenchimento e localização geográfica. Disparidades na AUC-PR por raça podem refletir disparidades no sinal disponível no registro (e não necessariamente viés algorítmico direto), o que não elimina a preocupação — apenas muda a intervenção necessária.

## Síntese

### Resultados quantitativos

| Métrica | Resultado |
|---|---|
| **AUC-PR (primária)** | 0,5531 — 5,5 vezes acima do baseline aleatório (cerca de 0,10) |
| **AUC-ROC** | 0,9055 — excelente discriminação geral |
| **Brier Score** | Calculado na seção 7.4 |
| **Threshold Youden** | Maximiza sensibilidade + especificidade; ponto de equilíbrio |
| **Threshold Recall >= 70%** | Piso clínico: aceita mais falsos positivos para não perder óbitos |
| **Calibração** | Razoável nos decis intermediários; extremos mostram desvio típico de modelos não calibrados |

### Features mais relevantes (SHAP — em linha com a EDA)
1. **SUPORT_VEN_ORD** — suporte ventilatório invasivo é o sinal clínico mais forte (confirmado pela EDA: 40,95% vs 4,07% de letalidade)
2. **UTI** — internação em UTI (3,2 vezes a letalidade de não-UTI)
3. **IDADE_ANOS** — gradiente monotônico de risco com a idade (confirmado na Fase 3)
4. Demais features listadas no beeswarm (seção 7.5)

### Equidade por subgrupo
- **Faixa etária:** AUC-PR menor em crianças (0-4 anos) — poucos óbitos, perfil atípico em relação ao dataset majoritariamente adulto/idoso
- **Região/Raça:** disparidades esperadas por variação na qualidade de preenchimento das fichas SIVEP

### Próxima fase
**Notebook 08 — Conclusões e Recomendações de Saúde Pública.**